In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
class WindowDataset(Dataset):
    def __init__(self,_data,_window):
        #_data : tensor, array
        #_window : 구간의 크기
        self.data=_data
        self.window=_window
        #DataLoader에서 사용이 가능한 인덱스의 최대값 
        self.n=len(_data) - _window

    def __len__(self):
        return self.n

    def __getitem__(self,idx):
        #idx : 0 ~ self.n-1 사이의 정수를 대입 (DataLoader에서 자동으로 대입)
        x = self.data[idx:idx + self.window]
        y = self.data[idx + self.window]
        return x, y


In [17]:
# RNN 모델 정의 
class RNNModel(nn.Module):
    def __init__(self,
                 input_size, 
                 hidden_size = 64, 
                 num_layers = 1, 
                 dropout = 0.0, 
                 nonlinearity = 'tanh', 
                 bidirectional = False):
        super(RNNModel, self).__init__()
        # super().__init__()
        self.rnn = nn.RNN(
            input_size = input_size,
            hidden_size = hidden_size, 
            num_layers = num_layers, 
            dropout = dropout, 
            nonlinearity = nonlinearity, 
            bidirectional = bidirectional, 
            # batch_first는 기본이 Fasle였던걸 까먹고 안 넣고 있었네요 
            # 죄송합니다. 
            # batch_first가 False인 경우, 입력 데이터(구간의 개수, 배치 크기, 입력 데이터 feature의 수) 
            # (배치 크기, 구간의 쿠기, 입력 feature의 수) -> True로 변경 
            batch_first = True
        )

        # output_feature가 역방향을 포함한다면 2배로 늘어난다. 
        if bidirectional:
            hidden_size *= 2
        print(f"hidden_size : {hidden_size}")
        self.model = nn.Linear(hidden_size, 1)
    def forward(self, x):
        out, h_n = self.rnn(x)
        last_hidden = h_n[-1]
        result = self.model(last_hidden)
        return result

In [18]:
#모델 학습 시 검증 데이터를 이용하여 모델의 성능을 평가할 수 있도록 검증 데이터 평가 함수

@torch.no_grad()
def evaluate_mse(dataloader,model):
    #dataloader : 검증 데이터셋의 dataloader
    model.eval()
    total_loss=0
    total_n=0
    for x, y in dataloader:
        x=x.float()
        y=y.float()
        pred=model(x)
        loss=nn.MSELoss()(pred,y)
        total_loss+=loss.item()*x.size(0)
        total_n+=x.size(0)
    return total_loss / max(total_n,1)


In [19]:
df=pd.read_csv('../../csv/AAPL.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9715 entries, 0 to 9714
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Date       9715 non-null   object 
 1   Open       9714 non-null   float64
 2   High       9714 non-null   float64
 3   Low        9714 non-null   float64
 4   Close      9714 non-null   float64
 5   Adj Close  9714 non-null   float64
 6   Volume     9713 non-null   float64
dtypes: float64(6), object(1)
memory usage: 531.4+ KB


In [20]:
df.dropna(inplace=True)

In [21]:
#수정종가, 날짜 데이터만 추출
df=df[['Date','Adj Close']]

In [22]:
values=df[['Adj Close']].values
values

array([[  0.410525],
       [  0.389106],
       [  0.360548],
       ...,
       [199.460007],
       [198.779999],
       [199.169998]], shape=(9713, 1))

In [23]:
df['Adj Close'].values.reshape(-1,1)

array([[  0.410525],
       [  0.389106],
       [  0.360548],
       ...,
       [199.460007],
       [198.779999],
       [199.169998]], shape=(9713, 1))

In [24]:
#75 : 25 비율로 학습, 검증 데이터 분할
split_idx=int(len(values)*0.75)
split_idx

7284

In [25]:
train_data=values[:split_idx]
test_data=values[split_idx:]

In [26]:
#MinMaxScaler로 데잍 정규화
scaler=MinMaxScaler()
train_sc=scaler.fit_transform(train_data)
test_sc=scaler.transform(test_data)

test_sc

array([[0.93723805],
       [0.956606  ],
       [0.91811525],
       ...,
       [7.80873011],
       [7.78208726],
       [7.7973675 ]], shape=(2429, 1))

In [27]:
#스케일링이 완료된 데이터를 Tensor로 변환
train_sc = torch.tensor(train_sc, dtype=torch.float32)
test_sc = torch.tensor(test_sc, dtype=torch.float32)

In [28]:
#WindowDataset에 데이터 대입
train_ds=WindowDataset(train_sc,_window=60)
test_ds=WindowDataset(test_sc,_window=60)

In [29]:
#Dataset을 DataLoader로 생성
train_dl=DataLoader(train_ds, batch_size=128, shuffle=True, drop_last=True)
test_dl=DataLoader(test_ds, batch_size=128, shuffle=False, drop_last=False)

In [30]:
aapl_model = RNNModel(
    input_size = 1
)
criterion = nn.MSELoss()
optimizer = optim.Adam(aapl_model.parameters(), lr = 0.001)

hidden_size : 64


In [31]:
# 모델 학습 
train_history, test_history = [], []
for epoch in range(20):
    aapl_model.train()
    running, n_seen = 0.0, 0
    for x, y in train_dl:
        x = x.float()
        y = y.float()
        pred = aapl_model(x)
        loss = criterion(pred, y)
        # print(x.shape, y.shape)
        # print(pred.shape, y.shape)
        # break
        optimizer.zero_grad()
        loss.backward()
        # 가중치 발산 방지 
        nn.utils.clip_grad_norm_(aapl_model.parameters(), 1.0)
        optimizer.step()

        running += loss.item() * y.size(0)
        n_seen += y.size(0)
    train_mse = running / n_seen
    test_mse = evaluate_mse(test_dl, aapl_model)
    train_history.append(train_mse)
    test_history.append(test_mse)
    print(f"Epoch {epoch+1} Train MSE: {round(train_mse, 8)} Test MSE: {round(test_mse, 8)}")

Epoch 1 Train MSE: 0.01228241 Test MSE: 0.0
Epoch 2 Train MSE: 0.00017338 Test MSE: 0.0
Epoch 3 Train MSE: 0.00012504 Test MSE: 0.0
Epoch 4 Train MSE: 0.00011621 Test MSE: 0.0
Epoch 5 Train MSE: 0.00010857 Test MSE: 0.0
Epoch 6 Train MSE: 0.00010913 Test MSE: 0.0
Epoch 7 Train MSE: 9.744e-05 Test MSE: 0.0
Epoch 8 Train MSE: 0.00010352 Test MSE: 0.0
Epoch 9 Train MSE: 0.00010821 Test MSE: 0.0
Epoch 10 Train MSE: 9.039e-05 Test MSE: 0.0
Epoch 11 Train MSE: 9.306e-05 Test MSE: 0.0
Epoch 12 Train MSE: 7.843e-05 Test MSE: 0.0
Epoch 13 Train MSE: 8.298e-05 Test MSE: 0.0
Epoch 14 Train MSE: 9.826e-05 Test MSE: 0.0
Epoch 15 Train MSE: 9.355e-05 Test MSE: 0.0
Epoch 16 Train MSE: 8.843e-05 Test MSE: 0.0
Epoch 17 Train MSE: 8.364e-05 Test MSE: 0.0
Epoch 18 Train MSE: 7.686e-05 Test MSE: 0.0
Epoch 19 Train MSE: 8.785e-05 Test MSE: 0.0
Epoch 20 Train MSE: 8.743e-05 Test MSE: 0.0
